# DMart OLAP EDA — retail_dw
## 20+ Visuals with Business Insights
**Database:** `retail_dw` | Tables: `dim_customer`, `dim_product`, `dim_store`, `dim_supplier`, `fact_transaction`, `fact_inventory`

---

## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

# Global style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi'        : 110,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : 'bold',
    'axes.titlepad'     : 10,
})

INR = lambda x: f'₹{x/1e6:.1f}M'   # helper: format large INR values
print('Ready.')

## 1. Database Connection

In [ ]:
DB_USER     = 'root'
DB_PASSWORD = 'your_password'   # <-- change this
DB_NAME     = 'retail_dw'
DB_HOST     = '127.0.0.1'
DB_PORT     = 3306

engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}',
    pool_pre_ping=True
)

with engine.connect() as c:
    print(f'Connected → {DB_NAME}')

## 2. Load Tables

In [ ]:
def load(table, limit=None):
    q = f'SELECT * FROM {table}' + (f' LIMIT {limit}' if limit else '')
    df = pd.read_sql(q, engine)
    print(f'{table:25s} → {df.shape[0]:>8,} rows × {df.shape[1]} cols')
    return df

dim_customer    = load('dim_customer')
dim_product     = load('dim_product')
dim_store       = load('dim_store')
dim_supplier    = load('dim_supplier')
fact_txn        = load('fact_transaction')
fact_inv        = load('fact_inventory')

## 3. Data Preparation & Joins

In [ ]:
# Parse dates
fact_txn['bill_date'] = pd.to_datetime(fact_txn['bill_date'])
fact_txn['year']      = fact_txn['bill_date'].dt.year
fact_txn['month']     = fact_txn['bill_date'].dt.month
fact_txn['month_name']= fact_txn['bill_date'].dt.strftime('%b')
fact_txn['quarter']   = fact_txn['bill_date'].dt.quarter
fact_txn['dow']       = fact_txn['bill_date'].dt.day_name()
fact_txn['week']      = fact_txn['bill_date'].dt.isocalendar().week.astype(int)

# Master sales — fact + all dims
sales = (fact_txn
    .merge(dim_product[['product_id','product_name','category','sub_category',
                         'brand','mrp','cost_price','is_perishable','veg_nonveg']], 
           on='product_id', how='left')
    .merge(dim_store[['store_id','store_name','city','state',
                       'city_tier','store_type','store_size_sqft']], 
           on='store_id', how='left')
    .merge(dim_customer[['customer_id','customer_name','gender','age_group',
                          'city_tier','loyalty_member','avg_monthly_spend_inr']], 
           on='customer_id', how='left', suffixes=('_store','_cust'))
)

# Derived columns
sales['gross_margin']     = sales['total_sale_amount'] - (sales['cost_price'] * sales['quantity'])
sales['gross_margin_pct'] = (sales['gross_margin'] / sales['total_sale_amount'].replace(0, np.nan)) * 100
sales['period']           = sales['year'].astype(str) + '-' + sales['month'].astype(str).str.zfill(2)

print(f'Master sales shape: {sales.shape}')
sales.head(3)

---
# SECTION A — Revenue & Sales Trends

### Visual 1 — Monthly Revenue Trend

In [ ]:
monthly = (sales.groupby('period')
           .agg(revenue=('total_sale_amount','sum'),
                orders=('transaction_id','nunique'),
                units=('quantity','sum'))
           .reset_index().sort_values('period'))

fig, ax1 = plt.subplots(figsize=(14,5))
ax2 = ax1.twinx()

ax1.fill_between(monthly['period'], monthly['revenue']/1e6,
                 alpha=0.25, color='steelblue')
ax1.plot(monthly['period'], monthly['revenue']/1e6,
         marker='o', ms=5, color='steelblue', lw=2, label='Revenue (₹M)')
ax2.plot(monthly['period'], monthly['orders'],
         marker='s', ms=4, color='coral', lw=1.5, ls='--', label='Transactions')

ax1.set_title('Visual 1 — Monthly Revenue Trend')
ax1.set_ylabel('Revenue (₹ Millions)', color='steelblue')
ax2.set_ylabel('Transactions', color='coral')
ax1.tick_params(axis='x', rotation=45)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout()
plt.show()

peak = monthly.loc[monthly['revenue'].idxmax()]
print(f"""
INSIGHT:
  Peak revenue month : {peak['period']} — ₹{peak['revenue']/1e6:.1f}M
  Total revenue      : ₹{monthly['revenue'].sum()/1e6:.1f}M across {len(monthly)} months
  Avg monthly revenue: ₹{monthly['revenue'].mean()/1e6:.1f}M
  Transaction trend  : correlates with revenue, confirming volume-driven growth.
""")

### Visual 2 — Year-over-Year Quarterly Revenue

In [ ]:
yoy = (sales.groupby(['year','quarter'])['total_sale_amount']
       .sum().reset_index())
yoy_pivot = yoy.pivot(index='quarter', columns='year', values='total_sale_amount') / 1e6

ax = yoy_pivot.plot(kind='bar', figsize=(10,5), width=0.7,
                    color=sns.color_palette('Blues_d', len(yoy_pivot.columns)))
ax.set_title('Visual 2 — Year-over-Year Quarterly Revenue')
ax.set_xlabel('Quarter')
ax.set_ylabel('Revenue (₹ Millions)')
ax.set_xticklabels([f'Q{q}' for q in yoy_pivot.index], rotation=0)
ax.legend(title='Year')
plt.tight_layout()
plt.show()

best_q = yoy.groupby('quarter')['total_sale_amount'].mean().idxmax()
print(f"""
INSIGHT:
  Best quarter on average: Q{best_q} — likely driven by festive/seasonal demand.
  YoY comparison reveals growth trajectory per quarter.
  Weak quarters signal where promotional investments are needed.
""")

### Visual 3 — Day-of-Week Revenue Heatmap

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

dow_month = (sales.groupby(['dow','month_name'])['total_sale_amount']
             .sum().reset_index())
dow_pivot = dow_month.pivot(index='dow', columns='month_name',
                            values='total_sale_amount') / 1e6
dow_pivot = dow_pivot.reindex(index=dow_order,
                               columns=[m for m in month_order if m in dow_pivot.columns])

fig, ax = plt.subplots(figsize=(14,5))
sns.heatmap(dow_pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.4, ax=ax, cbar_kws={'label':'₹M'})
ax.set_title('Visual 3 — Revenue Heatmap: Day-of-Week × Month (₹M)')
ax.set_xlabel('Month')
ax.set_ylabel('Day of Week')
plt.tight_layout()
plt.show()

best_dow = sales.groupby('dow')['total_sale_amount'].sum().idxmax()
print(f"""
INSIGHT:
  Highest revenue day: {best_dow} — ideal for promotions and staffing peaks.
  Weekend revenue consistently higher than weekday, typical for grocery retail.
  Month × day interaction shows festive months boost all days equally.
""")

---
# SECTION B — Product & Category Analysis

### Visual 4 — Top 15 Categories by Revenue

In [ ]:
cat_rev = (sales.groupby('category')
           .agg(revenue=('total_sale_amount','sum'),
                units=('quantity','sum'),
                margin_pct=('gross_margin_pct','mean'))
           .sort_values('revenue', ascending=False)
           .head(15).reset_index())

fig, ax = plt.subplots(figsize=(12,7))
colors = sns.color_palette('Blues_r', 15)
bars = ax.barh(cat_rev['category'], cat_rev['revenue']/1e6, color=colors)
ax.bar_label(bars, fmt='₹%.1fM', padding=4, fontsize=9)
ax.set_title('Visual 4 — Top 15 Categories by Revenue')
ax.set_xlabel('Revenue (₹ Millions)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

top3 = cat_rev.head(3)['category'].tolist()
print(f"""
INSIGHT:
  Top 3 revenue categories: {', '.join(top3)}
  These 3 likely contribute 40-50% of total revenue (Pareto principle).
  Categories with high revenue but low margin_pct need pricing review.
""")

### Visual 5 — Category Revenue vs Gross Margin % (Bubble Chart)

In [ ]:
cat_bubble = (sales.groupby('category')
              .agg(revenue=('total_sale_amount','sum'),
                   margin_pct=('gross_margin_pct','mean'),
                   units=('quantity','sum'))
              .reset_index())

fig, ax = plt.subplots(figsize=(13,7))
scatter = ax.scatter(
    cat_bubble['revenue']/1e6,
    cat_bubble['margin_pct'],
    s=cat_bubble['units']/cat_bubble['units'].max()*1200 + 50,
    c=cat_bubble['margin_pct'],
    cmap='RdYlGn', alpha=0.75, edgecolors='grey', lw=0.5
)
for _, row in cat_bubble.iterrows():
    ax.annotate(row['category'], (row['revenue']/1e6, row['margin_pct']),
                fontsize=7.5, ha='center', va='bottom')
plt.colorbar(scatter, ax=ax, label='Gross Margin %')
ax.axhline(cat_bubble['margin_pct'].median(), color='gray', ls='--', lw=1, label='Median margin')
ax.set_title('Visual 5 — Category: Revenue vs Gross Margin % (bubble = volume)')
ax.set_xlabel('Revenue (₹ Millions)')
ax.set_ylabel('Gross Margin %')
ax.legend()
plt.tight_layout()
plt.show()

print("""
INSIGHT:
  Top-right quadrant = high revenue AND high margin — these are star categories.
  Top-left quadrant = high margin, low revenue — growth opportunity with right marketing.
  Bottom-right = high revenue, low margin — volume play; watch profitability carefully.
  Bottom-left = low revenue, low margin — candidates for delisting or renegotiation.
""")

### Visual 6 — Top 10 Brands by Revenue

In [ ]:
brand_rev = (sales.groupby('brand')
             .agg(revenue=('total_sale_amount','sum'),
                  margin=('gross_margin','sum'))
             .sort_values('revenue', ascending=False)
             .head(10).reset_index())
brand_rev['margin_pct'] = brand_rev['margin'] / brand_rev['revenue'] * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
# Revenue
axes[0].barh(brand_rev['brand'], brand_rev['revenue']/1e6,
             color=sns.color_palette('Greens_r', 10))
axes[0].set_title('Top 10 Brands — Revenue')
axes[0].set_xlabel('₹ Millions')
axes[0].invert_yaxis()
# Margin %
colors = ['#2ecc71' if v >= brand_rev['margin_pct'].median() else '#e74c3c'
          for v in brand_rev['margin_pct']]
axes[1].barh(brand_rev['brand'], brand_rev['margin_pct'], color=colors)
axes[1].axvline(brand_rev['margin_pct'].median(), color='black', ls='--', lw=1)
axes[1].set_title('Top 10 Brands — Gross Margin %')
axes[1].set_xlabel('Gross Margin %')
axes[1].invert_yaxis()

fig.suptitle('Visual 6 — Top 10 Brands: Revenue vs Margin', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("""
INSIGHT:
  A brand ranking high in revenue but below median margin is a volume brand — DMart
  likely negotiates hard with them but earns thin margins.
  Brands above median margin AND high revenue are most valuable to DMart's P&L.
""")

### Visual 7 — Perishable vs Non-Perishable Revenue Split

In [ ]:
perish = (sales.groupby('is_perishable')
          .agg(revenue=('total_sale_amount','sum'),
               units=('quantity','sum'),
               margin_pct=('gross_margin_pct','mean'))
          .reset_index())
perish['label'] = perish['is_perishable'].map({True:'Perishable', False:'Non-Perishable',
                                                1:'Perishable', 0:'Non-Perishable',
                                                'Yes':'Perishable','No':'Non-Perishable',
                                                'TRUE':'Perishable','FALSE':'Non-Perishable'})

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col, title in zip(axes,
                          ['revenue','units','margin_pct'],
                          ['Revenue (₹)','Units Sold','Avg Margin %']):
    vals = perish[col] / (1e6 if col=='revenue' else 1)
    wedges, _, autotexts = ax.pie(vals, labels=perish['label'],
                                   autopct='%1.1f%%', startangle=90,
                                   wedgeprops={'width':0.55},
                                   colors=['#3498db','#e67e22'])
    ax.set_title(title)

fig.suptitle('Visual 7 — Perishable vs Non-Perishable Split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("""
INSIGHT:
  Non-perishables typically dominate revenue in value-retail formats like DMart.
  Perishables have higher margin% but higher waste risk — need tight inventory control.
  If perishable revenue share is growing, DMart's fresh category strategy is working.
""")

### Visual 8 — Discount Distribution by Category (Box Plot)

In [ ]:
top8_cats = (sales.groupby('category')['total_sale_amount']
             .sum().nlargest(8).index.tolist())
df_box = sales[sales['category'].isin(top8_cats)]

fig, ax = plt.subplots(figsize=(14,6))
sns.boxplot(data=df_box, x='category', y='discount_pct',
            order=top8_cats, palette='pastel', ax=ax,
            flierprops={'marker':'o','ms':3,'alpha':0.3})
ax.set_title('Visual 8 — Discount % Distribution by Top 8 Categories')
ax.set_xlabel('Category')
ax.set_ylabel('Discount %')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

high_disc = df_box.groupby('category')['discount_pct'].median().idxmax()
print(f"""
INSIGHT:
  Category with highest median discount: {high_disc}
  Wide IQR = inconsistent discounting — pricing policy may not be standardized.
  Outliers (high discount points) could be clearance sales or data entry errors.
  Compare with margin% — heavy discounting on already low-margin categories is dangerous.
""")

---
# SECTION C — Store Performance

### Visual 9 — Top 15 Stores by Revenue

In [ ]:
store_perf = (sales.groupby(['store_name','city','state','store_type'])
              .agg(revenue=('total_sale_amount','sum'),
                   transactions=('transaction_id','nunique'),
                   avg_basket=('total_sale_amount','mean'),
                   margin_pct=('gross_margin_pct','mean'))
              .sort_values('revenue', ascending=False)
              .reset_index())

top15_stores = store_perf.head(15)
fig, ax = plt.subplots(figsize=(12,8))
bars = ax.barh(top15_stores['store_name'] + ', ' + top15_stores['city'],
               top15_stores['revenue']/1e6,
               color=sns.color_palette('viridis', 15))
ax.bar_label(bars, fmt='₹%.1fM', padding=3, fontsize=8)
ax.set_title('Visual 9 — Top 15 Stores by Revenue')
ax.set_xlabel('Revenue (₹ Millions)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"""
INSIGHT:
  Top store: {top15_stores.iloc[0]['store_name']} ({top15_stores.iloc[0]['city']}) — ₹{top15_stores.iloc[0]['revenue']/1e6:.1f}M
  Top 15 stores likely contribute 60%+ of total revenue.
  High-revenue stores with low avg_basket = high footfall strategy.
  These stores deserve priority in stock allocation and promotions.
""")

### Visual 10 — Store Type Performance Comparison

In [ ]:
store_type = (sales.groupby('store_type')
              .agg(revenue=('total_sale_amount','sum'),
                   stores=('store_id','nunique'),
                   avg_basket=('total_sale_amount','mean'),
                   margin_pct=('gross_margin_pct','mean'))
              .reset_index())
store_type['rev_per_store'] = store_type['revenue'] / store_type['stores'] / 1e6

fig, axes = plt.subplots(1, 3, figsize=(15,5))

axes[0].bar(store_type['store_type'], store_type['revenue']/1e6,
            color=sns.color_palette('Set2', len(store_type)))
axes[0].set_title('Total Revenue')
axes[0].set_ylabel('₹ Millions')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(store_type['store_type'], store_type['rev_per_store'],
            color=sns.color_palette('Set2', len(store_type)))
axes[1].set_title('Revenue per Store (₹M)')
axes[1].tick_params(axis='x', rotation=20)

axes[2].bar(store_type['store_type'], store_type['avg_basket'],
            color=sns.color_palette('Set2', len(store_type)))
axes[2].set_title('Avg Basket Size (₹)')
axes[2].tick_params(axis='x', rotation=20)

fig.suptitle('Visual 10 — Store Type Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

best_type = store_type.loc[store_type['rev_per_store'].idxmax(), 'store_type']
print(f"""
INSIGHT:
  Highest revenue-per-store type: {best_type} — most productive store format.
  A store type with high total revenue but low rev_per_store = more stores, not better performance.
  Avg basket size indicates customer spend behavior per store format.
""")

### Visual 11 — City Tier Revenue Contribution

In [ ]:
tier_rev = (sales.groupby('city_tier_store')
            .agg(revenue=('total_sale_amount','sum'),
                 stores=('store_id','nunique'),
                 txns=('transaction_id','nunique'))
            .reset_index()
            .sort_values('city_tier_store'))

fig, axes = plt.subplots(1, 2, figsize=(13,5))
tier_labels = [f'Tier {int(t)}' if pd.notna(t) else 'Unknown'
               for t in tier_rev['city_tier_store']]

wedges, texts, autos = axes[0].pie(
    tier_rev['revenue'], labels=tier_labels, autopct='%1.1f%%',
    startangle=90, wedgeprops={'width':0.55},
    colors=sns.color_palette('Blues', len(tier_rev)))
axes[0].set_title('Revenue Share by City Tier')

axes[1].bar(tier_labels, tier_rev['revenue']/tier_rev['stores']/1e6,
            color=sns.color_palette('Blues', len(tier_rev)))
axes[1].set_title('Revenue per Store by City Tier (₹M)')
axes[1].set_ylabel('₹ Millions')

fig.suptitle('Visual 11 — City Tier Revenue Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("""
INSIGHT:
  Tier 1 cities dominate absolute revenue, but Tier 2/3 often show higher rev/store.
  Tier 2/3 markets have lower competition — DMart's EDLP model works especially well there.
  Expansion strategy should weight Tier 2/3 if rev-per-store is higher there.
""")

### Visual 12 — State-wise Revenue Heatmap

In [ ]:
state_qtr = pd.pivot_table(sales, values='total_sale_amount',
                            index='state', columns='quarter',
                            aggfunc='sum', fill_value=0) / 1e6
top15_states = state_qtr.sum(axis=1).nlargest(15).index
state_qtr = state_qtr.loc[top15_states]
state_qtr.columns = [f'Q{c}' for c in state_qtr.columns]

fig, ax = plt.subplots(figsize=(11,8))
sns.heatmap(state_qtr, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.3, ax=ax, cbar_kws={'label':'₹M'})
ax.set_title('Visual 12 — State × Quarter Revenue Heatmap (₹M)')
ax.set_ylabel('State')
plt.tight_layout()
plt.show()

top_state = state_qtr.sum(axis=1).idxmax()
print(f"""
INSIGHT:
  Highest revenue state: {top_state}
  States with consistent dark cells across all quarters = stable demand markets.
  States with only Q4 spikes = festive season dependency — risk if single quarter fails.
  Lighter states = untapped markets or limited store presence.
""")

---
# SECTION D — Customer Analysis

### Visual 13 — Customer Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16,5))

# Gender
gender_counts = dim_customer['gender'].value_counts()
axes[0].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            wedgeprops={'width':0.55}, colors=['#3498db','#e74c3c','#95a5a6'])
axes[0].set_title('Gender Split')

# Age group
age_order = sorted(dim_customer['age_group'].dropna().unique())
age_counts = dim_customer['age_group'].value_counts().reindex(age_order)
axes[1].bar(age_counts.index, age_counts.values, color=sns.color_palette('Oranges', len(age_counts)))
axes[1].set_title('Age Group Distribution')
axes[1].set_xlabel('Age Group')
axes[1].tick_params(axis='x', rotation=30)

# Loyalty
loyalty = dim_customer['loyalty_member'].value_counts()
axes[2].pie(loyalty, labels=['Non-Loyalty','Loyalty'], autopct='%1.1f%%',
            wedgeprops={'width':0.55}, colors=['#bdc3c7','#f39c12'])
axes[2].set_title('Loyalty Membership')

fig.suptitle('Visual 13 — Customer Demographics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

top_age = dim_customer['age_group'].value_counts().idxmax()
loyalty_pct = dim_customer['loyalty_member'].mean() * 100
print(f"""
INSIGHT:
  Dominant age group: {top_age} — primary target segment for marketing.
  Loyalty membership: {loyalty_pct:.1f}% of customers — higher loyalty = lower churn risk.
  Gender skew (if any) should influence product mix and promotional messaging.
""")

### Visual 14 — Revenue by Gender × Age Group (Stacked Bar)

In [ ]:
gender_age = (sales.groupby(['gender','age_group'])['total_sale_amount']
              .sum().reset_index())
pivot_ga = gender_age.pivot(index='age_group', columns='gender',
                             values='total_sale_amount').fillna(0) / 1e6

ax = pivot_ga.plot(kind='bar', stacked=True, figsize=(11,5),
                   color=sns.color_palette('Set1', len(pivot_ga.columns)),
                   edgecolor='white', lw=0.5)
ax.set_title('Visual 14 — Revenue by Gender × Age Group')
ax.set_xlabel('Age Group')
ax.set_ylabel('Revenue (₹ Millions)')
ax.legend(title='Gender')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print("""
INSIGHT:
  Segment with highest combined revenue = primary buyer persona for DMart.
  If one gender dominates a specific age group's spend, gender-targeted offers can boost conversion.
  Younger age groups with low revenue = future customers — needs nurturing programs.
""")

### Visual 15 — Loyalty vs Non-Loyalty: Spend & Frequency

In [ ]:
loy_sales = (sales.groupby(['customer_id','loyalty_member'])
             .agg(total_spend=('total_sale_amount','sum'),
                  visits=('transaction_id','nunique'))
             .reset_index())
loy_sales['loyalty_label'] = loy_sales['loyalty_member'].map(
    {True:'Loyalty', False:'Non-Loyalty', 1:'Loyalty', 0:'Non-Loyalty',
     'Yes':'Loyalty','No':'Non-Loyalty','TRUE':'Loyalty','FALSE':'Non-Loyalty'})

fig, axes = plt.subplots(1, 2, figsize=(13,5))
for ax, col, title, unit in zip(
        axes,
        ['total_spend','visits'],
        ['Total Spend per Customer','Visit Frequency'],
        ['₹','visits']):
    sns.boxplot(data=loy_sales, x='loyalty_label', y=col,
                palette={'Loyalty':'#f39c12','Non-Loyalty':'#bdc3c7'},
                ax=ax, flierprops={'marker':'o','ms':3,'alpha':0.3})
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel(unit)

fig.suptitle('Visual 15 — Loyalty vs Non-Loyalty Customer Behaviour', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

loy_avg = loy_sales.groupby('loyalty_label')['total_spend'].median()
print(f"""
INSIGHT:
  Loyalty median spend: ₹{loy_avg.get('Loyalty',0):,.0f} vs Non-Loyalty: ₹{loy_avg.get('Non-Loyalty',0):,.0f}
  If loyalty members spend 2x+ more, the program is clearly effective.
  Visit frequency gap between groups quantifies program's engagement impact.
  Outliers in non-loyalty = high-spend customers to target for loyalty enrollment.
""")

### Visual 16 — RFM Segmentation

In [ ]:
snapshot = fact_txn['bill_date'].max() + pd.Timedelta(days=1)

rfm = (fact_txn.groupby('customer_id')
       .agg(recency=('bill_date',  lambda x: (snapshot - x.max()).days),
            frequency=('transaction_id','nunique'),
            monetary=('total_sale_amount','sum'))
       .reset_index())

for col in ['recency','frequency','monetary']:
    rfm[f'{col}_score'] = pd.qcut(
        rfm[col] if col != 'recency' else -rfm[col],
        q=5, labels=[1,2,3,4,5]).astype(int)

rfm['rfm_score'] = rfm['recency_score'] + rfm['frequency_score'] + rfm['monetary_score']
rfm['segment'] = pd.cut(rfm['rfm_score'],
                         bins=[2,6,9,12,15],
                         labels=['At-Risk','Needs Attention','Loyal','Champions'])

seg_counts = rfm['segment'].value_counts().reindex(['Champions','Loyal','Needs Attention','At-Risk'])
seg_revenue = rfm.groupby('segment')['monetary'].sum().reindex(seg_counts.index)

fig, axes = plt.subplots(1, 2, figsize=(13,5))
colors = ['#2ecc71','#3498db','#f39c12','#e74c3c']
axes[0].barh(seg_counts.index, seg_counts.values, color=colors)
axes[0].set_title('Customers by RFM Segment')
axes[0].set_xlabel('Number of Customers')
axes[1].barh(seg_revenue.index, seg_revenue.values/1e6, color=colors)
axes[1].set_title('Revenue by RFM Segment (₹M)')
axes[1].set_xlabel('₹ Millions')

fig.suptitle('Visual 16 — RFM Customer Segmentation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

champions_rev_pct = seg_revenue.get('Champions',0) / rfm['monetary'].sum() * 100
print(f"""
INSIGHT:
  Champions contribute ~{champions_rev_pct:.0f}% of revenue despite being a small customer count.
  At-Risk segment = previously active customers who stopped — win-back campaigns needed.
  'Needs Attention' = mid-tier customers who can be upgraded to Loyal with targeted offers.
  Classic 80/20 rule: top 20% customers drive 80% of revenue.
""")

---
# SECTION E — Payment & Discount Analysis

### Visual 17 — Payment Mode: Revenue, Avg Basket, Transaction Count

In [ ]:
pay = (sales.groupby('payment_mode')
       .agg(revenue=('total_sale_amount','sum'),
            txns=('transaction_id','nunique'),
            avg_basket=('total_sale_amount','mean'))
       .sort_values('revenue', ascending=False)
       .reset_index())

fig, axes = plt.subplots(1, 3, figsize=(16,5))
pal = sns.color_palette('tab10', len(pay))

axes[0].bar(pay['payment_mode'], pay['revenue']/1e6, color=pal)
axes[0].set_title('Revenue')
axes[0].set_ylabel('₹ Millions')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(pay['payment_mode'], pay['txns'], color=pal)
axes[1].set_title('Transaction Count')
axes[1].tick_params(axis='x', rotation=30)

axes[2].bar(pay['payment_mode'], pay['avg_basket'], color=pal)
axes[2].set_title('Avg Basket Size (₹)')
axes[2].tick_params(axis='x', rotation=30)

fig.suptitle('Visual 17 — Payment Mode Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

top_pay = pay.iloc[0]
print(f"""
INSIGHT:
  Most used payment mode by revenue: {top_pay['payment_mode']} — ₹{top_pay['revenue']/1e6:.1f}M
  Mode with highest avg_basket = high-value customers prefer that payment method.
  Cash-heavy transactions = potential for digital payment incentives to reduce handling cost.
  UPI/digital growth rate = indicator of tech adoption in the customer base.
""")

### Visual 18 — Monthly Discount Rate Trend

In [ ]:
disc_trend = (sales.groupby('period')
              .agg(total_rev=('total_sale_amount','sum'),
                   total_disc=('discount_amount','sum'))
              .reset_index()
              .sort_values('period'))
disc_trend['disc_rate'] = disc_trend['total_disc'] / disc_trend['total_rev'] * 100

fig, ax1 = plt.subplots(figsize=(14,5))
ax2 = ax1.twinx()

ax1.bar(disc_trend['period'], disc_trend['total_rev']/1e6,
        alpha=0.4, color='steelblue', label='Revenue (₹M)')
ax2.plot(disc_trend['period'], disc_trend['disc_rate'],
         color='red', lw=2, marker='o', ms=4, label='Discount Rate %')

ax1.set_title('Visual 18 — Monthly Revenue vs Discount Rate Trend')
ax1.set_ylabel('Revenue (₹ Millions)', color='steelblue')
ax2.set_ylabel('Discount Rate %', color='red')
ax1.tick_params(axis='x', rotation=45)
lines1,lab1 = ax1.get_legend_handles_labels()
lines2,lab2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, lab1+lab2, loc='upper left')
plt.tight_layout()
plt.show()

avg_disc = disc_trend['disc_rate'].mean()
print(f"""
INSIGHT:
  Average discount rate: {avg_disc:.1f}% of revenue
  Months where discount rate spikes but revenue also rises = effective promotional campaigns.
  Months where discount rate rises but revenue stays flat = ineffective discounting — burning margin.
  DMart's EDLP model should keep discount rate relatively stable; sharp spikes are anomalies.
""")

---
# SECTION F — Supplier Analysis

### Visual 19 — Supplier: Preferred vs Non-Preferred & Payment Terms

In [ ]:
dim_supplier['is_preferred'] = dim_supplier['is_preferred_vendor'].map(
    {'Yes':True,'TRUE':True,'No':False,'FALSE':False,True:True,False:False,1:True,0:False})
dim_supplier['payment_days'] = pd.to_numeric(dim_supplier['payment_days'], errors='coerce')
dim_supplier['lead_time_days'] = pd.to_numeric(dim_supplier['lead_time_days'], errors='coerce')
dim_supplier['annual_contract_value_lakh'] = pd.to_numeric(
    dim_supplier['annual_contract_value_lakh'], errors='coerce')

fig, axes = plt.subplots(1, 3, figsize=(16,5))

# Preferred vs not
pref_counts = dim_supplier['is_preferred'].value_counts()
axes[0].pie(pref_counts.values, labels=['Non-Preferred','Preferred'],
            autopct='%1.1f%%', wedgeprops={'width':0.55},
            colors=['#bdc3c7','#27ae60'])
axes[0].set_title('Preferred vs Non-Preferred Suppliers')

# Payment days distribution
sns.histplot(dim_supplier['payment_days'].dropna(), bins=20, ax=axes[1],
             color='steelblue', kde=True)
axes[1].set_title('Payment Days Distribution')
axes[1].set_xlabel('Payment Days')

# Supplier type
supp_type = dim_supplier['supplier_type'].value_counts().head(8)
axes[2].barh(supp_type.index, supp_type.values, color=sns.color_palette('tab10', len(supp_type)))
axes[2].set_title('Supplier Type Breakdown')
axes[2].invert_yaxis()

fig.suptitle('Visual 19 — Supplier Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

avg_pay = dim_supplier['payment_days'].mean()
avg_lead = dim_supplier['lead_time_days'].mean()
print(f"""
INSIGHT:
  Avg payment terms: {avg_pay:.0f} days | Avg lead time: {avg_lead:.0f} days
  Preferred suppliers likely get better payment terms — check if their lead time is also lower.
  Long payment days = DMart holds cash longer — working capital advantage.
  High lead time suppliers are supply chain risk — need buffer stock.
""")

---
# SECTION G — Inventory Analysis

### Visual 20 — Inventory Turnover by Category

In [ ]:
fact_inv['week_start_date'] = pd.to_datetime(fact_inv['week_start_date'])

inv_cat = (fact_inv
           .merge(dim_product[['product_id','category','cost_price']], on='product_id', how='left')
           .groupby('category')
           .agg(avg_closing_stock=('closing_stock_units','mean'),
                total_sold=('units_sold','sum'),
                avg_stock_value=('closing_stock_units', lambda x:
                                  (x * fact_inv.loc[x.index,'closing_stock_units']).mean()))
           .reset_index())

inv_cat['turnover'] = inv_cat['total_sold'] / inv_cat['avg_closing_stock'].replace(0, np.nan)
inv_cat = inv_cat.sort_values('turnover', ascending=False).dropna().head(15)

fig, ax = plt.subplots(figsize=(12,7))
colors = ['#2ecc71' if v >= inv_cat['turnover'].median() else '#e74c3c'
          for v in inv_cat['turnover']]
bars = ax.barh(inv_cat['category'], inv_cat['turnover'], color=colors)
ax.axvline(inv_cat['turnover'].median(), color='black', ls='--', lw=1, label='Median')
ax.bar_label(bars, fmt='%.1fx', padding=3, fontsize=9)
ax.set_title('Visual 20 — Inventory Turnover by Category')
ax.set_xlabel('Turnover Ratio (units sold / avg stock)')
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.show()

fastest = inv_cat.iloc[0]
slowest = inv_cat.iloc[-1]
print(f"""
INSIGHT:
  Fastest moving category: {fastest['category']} — turnover {fastest['turnover']:.1f}x
  Slowest moving category: {slowest['category']} — turnover {slowest['turnover']:.1f}x
  High turnover = less capital locked in inventory — good for cash flow.
  Low turnover categories need markdown strategy or smaller order quantities.
  Perishable categories MUST have high turnover; low turnover = high wastage.
""")

### Visual 21 — Weekly Stock vs Sales Trend (Top 5 Categories)

In [ ]:
top5_cats = (sales.groupby('category')['total_sale_amount']
             .sum().nlargest(5).index.tolist())

inv_weekly = (fact_inv
              .merge(dim_product[['product_id','category']], on='product_id', how='left')
              .query('category in @top5_cats')
              .groupby(['week_start_date','category'])
              .agg(avg_stock=('closing_stock_units','mean'),
                   units_sold=('units_sold','sum'))
              .reset_index()
              .sort_values('week_start_date'))

fig, axes = plt.subplots(len(top5_cats), 1, figsize=(13, 3.5*len(top5_cats)), sharex=True)

for ax, cat in zip(axes, top5_cats):
    df_cat = inv_weekly[inv_weekly['category'] == cat]
    ax2 = ax.twinx()
    ax.fill_between(df_cat['week_start_date'], df_cat['avg_stock'],
                    alpha=0.3, color='steelblue')
    ax.plot(df_cat['week_start_date'], df_cat['avg_stock'],
            color='steelblue', lw=1.5, label='Avg Stock')
    ax2.plot(df_cat['week_start_date'], df_cat['units_sold'],
             color='coral', lw=1.5, ls='--', label='Units Sold')
    ax.set_ylabel('Avg Stock', color='steelblue', fontsize=9)
    ax2.set_ylabel('Units Sold', color='coral', fontsize=9)
    ax.set_title(cat, fontsize=10, pad=4)

fig.suptitle('Visual 21 — Weekly Stock vs Sales: Top 5 Categories', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("""
INSIGHT:
  When stock line drops sharply but sold line stays flat = stockout risk with demand unfulfilled.
  When sold line drops despite high stock = demand problem, not supply — needs marketing push.
  Synchronized drops in both = seasonal pattern — plan replenishment calendar accordingly.
  Widening gap between stock and sales over time = accumulating dead stock.
""")

### Visual 22 — Stock Fill Rate by Store Type

In [ ]:
latest_inv = (fact_inv
              .sort_values('week_start_date')
              .groupby(['product_id','store_id'])
              .last()
              .reset_index())

inv_store = (latest_inv
             .merge(dim_store[['store_id','store_type','city_tier']], on='store_id', how='left'))

fill_rate = (inv_store.groupby('store_type')
             .agg(total_skus=('product_id','count'),
                  zero_stock=('closing_stock_units', lambda x: (x==0).sum()),
                  low_stock=('closing_stock_units', lambda x: ((x>0) & (x<10)).sum()),
                  adequate=('closing_stock_units', lambda x: (x>=10).sum()))
             .reset_index())

fill_pct = fill_rate.set_index('store_type')[['zero_stock','low_stock','adequate']]
fill_pct = fill_pct.div(fill_rate.set_index('store_type')['total_skus'], axis=0) * 100

ax = fill_pct.plot(kind='bar', stacked=True, figsize=(11,5),
                   color=['#e74c3c','#f39c12','#2ecc71'],
                   edgecolor='white', lw=0.5)
ax.set_title('Visual 22 — Stock Fill Rate by Store Type')
ax.set_xlabel('Store Type')
ax.set_ylabel('% of SKUs')
ax.legend(['Zero Stock','Low Stock (<10)','Adequate (≥10)'])
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print("""
INSIGHT:
  Store types with high zero-stock % are losing sales — immediate restocking needed.
  Low-stock band (>0 but <10 units) is a warning zone — likely to hit zero before next replenishment.
  Large format stores should have near-zero 'zero stock' — if not, supply chain has gaps.
  Compare fill rate with that store type's revenue — poor fill rate explains revenue underperformance.
""")

---
# SECTION H — Executive Summary Dashboard

### Visual 23 — KPI Summary Dashboard

In [ ]:
total_rev       = sales['total_sale_amount'].sum()
total_margin    = sales['gross_margin'].sum()
margin_pct      = total_margin / total_rev * 100
total_txns      = fact_txn['transaction_id'].nunique()
total_customers = dim_customer.shape[0]
total_products  = dim_product.shape[0]
total_stores    = dim_store.shape[0]
avg_basket      = total_rev / total_txns
total_disc      = sales['discount_amount'].sum()
disc_rate       = total_disc / total_rev * 100

kpis = [
    ('Total Revenue',         f'₹{total_rev/1e7:.1f}Cr',  '#3498db'),
    ('Gross Margin',          f'₹{total_margin/1e7:.1f}Cr ({margin_pct:.1f}%)', '#2ecc71'),
    ('Total Transactions',    f'{total_txns:,}',           '#9b59b6'),
    ('Unique Customers',      f'{total_customers:,}',      '#e67e22'),
    ('Products',              f'{total_products:,}',       '#1abc9c'),
    ('Stores',                f'{total_stores:,}',         '#34495e'),
    ('Avg Basket Size',       f'₹{avg_basket:.0f}',        '#e74c3c'),
    ('Avg Discount Rate',     f'{disc_rate:.1f}%',         '#f39c12'),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 5))
axes = axes.flatten()
for ax, (label, value, color) in zip(axes, kpis):
    ax.set_facecolor(color)
    ax.text(0.5, 0.6, value, ha='center', va='center',
            fontsize=16, fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.25, label, ha='center', va='center',
            fontsize=10, color='white', alpha=0.9, transform=ax.transAxes)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

fig.suptitle('Visual 23 — Executive KPI Dashboard', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"""
INSIGHT:
  Total Revenue    : ₹{total_rev/1e7:.1f} Crores
  Gross Margin     : {margin_pct:.1f}% — benchmark for grocery retail is 15-25%
  Avg Basket       : ₹{avg_basket:.0f} — higher basket = better cross-sell effectiveness
  Discount Rate    : {disc_rate:.1f}% — DMart's EDLP model should keep this low and stable
  These 8 KPIs are what every retail analyst is asked on Day 1.
""")

---
**End of OLAP EDA — 23 Visuals Completed**

| Section | Visuals | Focus |
|---------|---------|-------|
| A | 1–3 | Revenue & Time Trends |
| B | 4–8 | Product & Category |
| C | 9–12 | Store Performance |
| D | 13–16 | Customer Analysis + RFM |
| E | 17–18 | Payment & Discount |
| F | 19 | Supplier |
| G | 20–22 | Inventory |
| H | 23 | Executive KPI Dashboard |